# 01 · Dataset overview

What does the dish dataset look like before training?

* how many images each class has (raw and after the train/val/test split)
* sample images per class — a quick visual check for wrong photos
* image sizes

Run from the repository root after `training/scrape.py` and `training/split_dataset.py`.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
SPLIT = ROOT / "data" / "split"
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def images(folder):
    return [f for f in folder.iterdir() if f.suffix.lower() in EXT] if folder.exists() else []

classes = sorted(p.name for p in RAW.iterdir() if p.is_dir())
print(len(classes), "classes:", ", ".join(classes))

## Images per class

In [ ]:
raw_counts = {c: len(images(RAW / c)) for c in classes}
split_counts = {part: {c: len(images(SPLIT / part / c)) for c in classes} for part in ("train", "val", "test")}

fig, ax = plt.subplots(figsize=(11, 4))
bottom = [0] * len(classes)
for part, color in (("train", "#2E8B57"), ("val", "#F2A900"), ("test", "#E5484D")):
    values = [split_counts[part][c] for c in classes]
    ax.bar(classes, values, bottom=bottom, label=part, color=color)
    bottom = [b + v for b, v in zip(bottom, values)]
ax.set_ylabel("images")
ax.set_title(f"Images per class after split — total {sum(bottom)}")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

for c in classes:
    print(f"{c:15s} raw={raw_counts[c]:4d}  train={split_counts['train'][c]:4d}  "
          f"val={split_counts['val'][c]:3d}  test={split_counts['test'][c]:3d}")

## Sample images per class

Look for photos that do not belong (logos, menus, people, other dishes) and delete them from `data/raw/`.

In [ ]:
random.seed(0)
per_class = 6
fig, axes = plt.subplots(len(classes), per_class, figsize=(per_class * 2, len(classes) * 2))
for row, c in enumerate(classes):
    files = images(RAW / c)
    sample = random.sample(files, min(per_class, len(files)))
    for col in range(per_class):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(sample):
            ax.imshow(Image.open(sample[col]).convert("RGB"))
        if col == 0:
            ax.set_title(c, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Image sizes

In [ ]:
sizes = []
for c in classes:
    for f in images(RAW / c):
        try:
            with Image.open(f) as im:
                sizes.append(im.size)
        except Exception:
            pass
widths, heights = zip(*sizes)
plt.figure(figsize=(5, 5))
plt.scatter(widths, heights, s=6, alpha=0.4)
plt.xlabel("width (px)"); plt.ylabel("height (px)")
plt.title(f"{len(sizes)} images — the model resizes all of them to 224×224")
plt.show()